# Damage maps of ions in SiC

Originally a production notebook for a [publication on coupled electronic and atomic effects
on defect evolution in silicon carbide under ion irradiation](https://doi.org/10.1016/j.cossms.2017.09.003). This version runs a few ions of two species with full
cascades and maps where their collisions happen. It replaces the former separate KTaO3
notebook, which did the same for another material: change the `Layer` below to reuse it.

The events come from `Collision.iter_events()`; the positions are accumulated while
streaming, so even a `COLLISON.txt` of many gigabytes needs little memory. For the
published figures use 10,000 ions per species; the small numbers here only show the method.
Set up SRIM once with `pysrim2 install` and check it with `pysrim2 doctor`
(see the [installation guide](https://pysrim2.readthedocs.io/en/latest/installation.html)). The notebook then finds SRIM
by itself, natively or in the container; no path is needed. SRIM-2013 only, **not** 2018.
SRIM is (c) SRIM.com (James F. Ziegler), free for non-commercial use only; see `LEGAL.rtf` next to your SRIM copy.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

from srim import TRIM, Ion, Layer, Target
from srim.paths import srim_dir

OUT = Path("sic-output")
OUT.mkdir(exist_ok=True)
NUM_IONS = 5  # raise for smooth maps (a full cascade of thousands of ions takes hours)

ions = [  # symbol, energy [eV]
    ("C", 4.5e6),
    ("Si", 6.0e6),
]
layer = Layer(
    {
        "Si": {"stoich": 0.5, "E_d": 35.0, "lattice": 0.0, "surface": 3.0},
        "C": {"stoich": 0.5, "E_d": 20.0, "lattice": 0.0, "surface": 3.0},
    },
    density=3.21,
    width=40000.0,  # 4 um
)
target = Target([layer])

## Run each ion, keep the collision positions

SRIM overwrites its output every run, so the events are read right after each run (and the raw
files are copied, `COLLISON.txt` included, if you want them).

In [ ]:
events = {}
for symbol, energy in ions:
    trim = TRIM(
        target, Ion(symbol, energy=energy), number_ions=NUM_IONS,
        calculation=2, collisions=2, random_seed=12345,
    )
    results = trim.run()
    df = results.collision.df
    df["ion_species"] = symbol
    events[symbol] = df
    folder = OUT / symbol
    folder.mkdir(exist_ok=True)
    TRIM.copy_output_files(srim_dir(), folder)
    print(symbol, len(df), "events")
allev = pd.concat(events.values(), ignore_index=True)
allev.groupby(["ion_species", "kind"]).size()

## Export everything as Parquet

One file with a column `ion_species` (needs `pysrim2[parquet]`).

In [ ]:
allev.to_parquet(OUT / "sic-collisions.parquet")

## Depth-versus-lateral map

Positions are in angstrom; `depth_A` is along the surface normal, `y_A` and `z_A` are
lateral. Log colour scale.

In [ ]:
fig, axes = plt.subplots(len(ions), 1, figsize=(10, 3.5 * len(ions)), squeeze=False)
for ax, (symbol, _) in zip(axes[:, 0], ions, strict=True):
    df = events[symbol]
    h = ax.hist2d(df["depth_A"], df["y_A"], bins=(200, 80), norm=mcolors.LogNorm())
    ax.set(title=symbol, xlabel="Depth [A]", ylabel="y [A]")
    fig.colorbar(h[3], ax=ax)
fig.tight_layout()
plt.show()

## Lateral (yz) map and radial distribution

In [ ]:
fig, axes = plt.subplots(1, len(ions), figsize=(6 * len(ions), 5), squeeze=False)
for ax, (symbol, _) in zip(axes[0], ions, strict=True):
    df = events[symbol]
    h = ax.hist2d(df["y_A"], df["z_A"], bins=100, norm=mcolors.LogNorm())
    ax.set(title=symbol, xlabel="y [A]", ylabel="z [A]")
    fig.colorbar(h[3], ax=ax)
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for symbol, _ in ions:
    df = events[symbol]
    r = np.hypot(df["y_A"], df["z_A"])
    ax.hist(r, bins=np.linspace(0, 1000, 100), histtype="step", label=symbol)
ax.set(xlabel="Radial distance from the track [A]", ylabel="collisions", yscale="log")
ax.legend()
plt.show()